# CuPyCCx — GPU test on Google Colab

**Before running:** set runtime to GPU via *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Cell 1 — confirm GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
%%bash
# Cell 2 — install dependencies; remove any stale cupyccx install
apt-get install -qq cmake ninja-build libeigen3-dev libopenblas-dev
pip install -q --upgrade pip
pip install -q pybind11 pyscf
pip uninstall -q -y cupyccx 2>/dev/null || true

In [ ]:
%%bash
# Cell 3 — clone and build with CUDA (T4 = sm_75; A100 = sm_80)
# Always start from /content so re-running doesn't nest CuPyCCx/CuPyCCx/...
cd /content
rm -rf CuPyCCx
git clone --quiet https://github.com/varunrishi/CuPyCCx.git
cd CuPyCCx
cmake -B build \
  -DCUPYCCX_CUDA=ON \
  -DCUPYCCX_CUDA_ARCH=75 \
  -DCMAKE_BUILD_TYPE=Release \
  -DCUPYCCX_BUILD_TESTS=OFF \
  -Dpybind11_DIR=$(python3 -c "import pybind11; print(pybind11.get_cmake_dir())")
cmake --build build -j$(nproc)

# Install Python files + CUDA extension directly into site-packages
SITE=$(python3 -c "import site; print(site.getsitepackages()[0])")
rm -rf "$SITE/cupyccx"
cp -r python/cupyccx "$SITE/"
cp build/_cupyccx*.so "$SITE/cupyccx/"
echo "Installed to: $SITE/cupyccx/"
ls "$SITE/cupyccx/"

# Verify import works before leaving bash
python3 -c "import importlib; importlib.invalidate_caches(); import cupyccx._cupyccx; print('Extension OK:', cupyccx._cupyccx.__file__)"

In [ ]:
# Cell 4 — verify extension loads in the notebook kernel
import sys, importlib, os, glob

# Evict all stale cupyccx modules
for key in list(sys.modules):
    if 'cupyccx' in key:
        del sys.modules[key]

# Find the installed .so and promote its site-packages to the front of sys.path.
# This handles the case where a stale editable install elsewhere on sys.path
# shadows the freshly built extension.
matches = glob.glob('/usr/local/lib/python*/dist-packages/cupyccx/_cupyccx*.so')
if matches:
    site_dir = os.path.dirname(os.path.dirname(matches[0]))
    sys.path = [site_dir] + [p for p in sys.path if p != site_dir]
    print(f'Using site-packages: {site_dir}')
else:
    print('WARNING: could not find _cupyccx*.so under /usr/local/lib — Cell 3 may not have completed')

importlib.invalidate_caches()

import cupyccx._cupyccx
print('Extension loaded from:', cupyccx._cupyccx.__file__)

In [ ]:
import time
from pyscf import gto, scf
from cupyccx.scf_data import prepare_from_pyscf
from cupyccx.method import CCD, CCOptions

# Cell 5 — N2/STO-3G: CPU vs GPU correctness check
mol  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='sto-3g', unit='Bohr', verbose=0)
mf   = scf.RHF(mol).run()
data = prepare_from_pyscf(mf, verbose=False)

t0 = time.time()
r_cpu = CCD.from_scf_data(data, opts=CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r_gpu = CCD.from_scf_data(data, opts=CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Diff        = {abs(r_cpu.e_corr - r_gpu.e_corr):.2e} Ha')

In [ ]:
# Cell 6 — larger system (cc-pVDZ) to see GPU speedup
mol2  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='cc-pVDZ', unit='Bohr', verbose=0)
mf2   = scf.RHF(mol2).run()
data2 = prepare_from_pyscf(mf2, verbose=False)
print(f'n_occ={data2.n_occ}  n_vir={data2.n_vir}  n_mo={data2.n_mo}')

t0 = time.time()
r2_cpu = CCD.from_scf_data(data2, opts=CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data2.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r2_gpu = CCD.from_scf_data(data2, opts=CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data2.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r2_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r2_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Speedup     = {t_cpu/t_gpu:.1f}x')